In [11]:
import numpy as np
from pathlib import Path
import os, sys
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm
import warnings
import pandas as pd

repo_root = Path("/home/thardy/elefanto/ConsciousnessTeam_Data/SOUNDMODEL/Data_SoundGOOD/LEAD_ExperimentalFolder")
os.chdir(repo_root)

if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))
os.environ["PYTHONPATH"] = str(repo_root) + os.pathsep + os.environ.get("PYTHONPATH", "")

import LEAD as lead


# Important variables
cwd = Path.cwd()
SNRs = np.array([-np.inf,-13,-11,-9,-7,-5,-3])
SubIDs = ['01','02','03','05','06','07','08','09','11','12','13','14','15','17','19','20','22','23','24','25']
colormap = {0: (0, 0, 0), 1: (0, 0.25, 1), 2: (0, 0.9375, 1), 3: (0, 0.91, 0.1), 4: (1, 0.6, 0), 5: (1, 0, 0), 6: (0.8, 0, 0)}

In [12]:
def metastability_score(f, df, x_grid, alpha=1.0):
    cost = f(x_grid)**2 + alpha * df(x_grid)**2   # alpha à calibrer
    return -np.min(cost)   # plus c'est grand, plus on est près d'un ghost

# Metastability Score

In [13]:
task = "Active"
period = 'early'
file_path = Path(f'Fit_{task}_{period}')

# Define the summary path clearly and load existing progress
summary_path = file_path / "metastability_scores.csv"
if summary_path.exists():
    print(f"Found existing summary at {summary_path}. Loading progress...")
    existing_df = pd.read_csv(summary_path, index_col=0)
    metastability_scores = existing_df["metastability_score"].to_dict()
    print(metastability_scores)
else:
    metastability_scores = {}

    
for part in range(20):
    print(f'part {part} lets go')

    # --- SKIP if already processed --- 
    if part in metastability_scores:
        print(f">>> Part {part} already exists in CSV. Skipping...")
        continue

    # --- Load Data ---
    data_ref = f'myEpochs_{task}/Epoch_{SubIDs[part]}-epo.fif'
    epochs_file = cwd.parents[0] / data_ref
    
    # --- Load MLE Parameters ---
    gainfixed_mle = lead.model.StratifiedNonLinear1(
        tau=10, process_noise=0.1, measure_noise=0.1, threshold=1, sharpness=5, gain=0)
    param_path = file_path / f"GainFixedModel_part{part}_params"
    gainfixed_mle.load_params(param_path)

    # --- Compute Metastability Score ---
    model_func = gainfixed_mle.core

    def f(states, input_value=0, signal_category=0):
        return model_func(states, input_value * np.ones_like(states), signal_category) - states
    
    def df(states, input_value=0, signal_category=0):
        epsilon = 1e-5
        return (f(states + epsilon, input_value, signal_category) - f(states - epsilon, input_value, signal_category)) / (2 * epsilon)
    
    max_metascore = -np.inf
    for signal_category in range(1, 7):
        x_grid = np.linspace(-0.5, 2, 1000)
        metascore = metastability_score(lambda x: f(x, input_value=1, signal_category=signal_category), 
                                        lambda x: df(x, input_value=1, signal_category=signal_category), 
                                        x_grid, alpha=1.0)
        max_metascore = max(max_metascore, metascore)

    metastability_scores[part] = max_metascore
    print(f"Part {part} - Metastability Score: {max_metascore}")

    # Save summary CSV
    df_results = pd.DataFrame.from_dict(
        metastability_scores, orient="index", columns=["metastability_score"]
    ).sort_index()
    df_results.to_csv(file_path / "metastability_scores.csv")

part 0 lets go
Part 0 - Metastability Score: -0.05246168822486964
part 1 lets go
Part 1 - Metastability Score: -0.00546764186044412
part 2 lets go
Part 2 - Metastability Score: -0.04678898052696163
part 3 lets go
Part 3 - Metastability Score: -0.008278974534730112
part 4 lets go
Part 4 - Metastability Score: -0.007642875632346614
part 5 lets go
Part 5 - Metastability Score: -0.011212766069063344
part 6 lets go
Part 6 - Metastability Score: -0.027749473522330034
part 7 lets go
Part 7 - Metastability Score: -0.01488504181118496
part 8 lets go
Part 8 - Metastability Score: -0.09494086970179207
part 9 lets go
Part 9 - Metastability Score: -0.08876832743021376
part 10 lets go
Part 10 - Metastability Score: -0.08750976534437369
part 11 lets go
Part 11 - Metastability Score: -0.0682654713449604
part 12 lets go
Part 12 - Metastability Score: -0.13929695259178937
part 13 lets go
Part 13 - Metastability Score: -1.8681394315663003e-05
part 14 lets go
Part 14 - Metastability Score: -0.010527762855